# 構造化出力とエージェンティックワークフロー


## Structured Outputs（構造化出力）


In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List


class Person(BaseModel):
    name: str
    age: int | None = Field(description="年齢、不明な場合はNone")


class PersonList(BaseModel):
    people: List[Person]


client = OpenAI()
response = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=[
        {
            "role": "developer",
            "content": "人物を抽出してください。",
        },
        {
            "role": "user",
            "content": "昔々あるところにおじいさん(70)とおばあさん(年齢不詳)がいました。",
        },
    ],
    reasoning_effort="minimal",
    response_format=PersonList,
)

parsed_result = response.choices[0].message.parsed
print(type(parsed_result))
print(parsed_result)

## with_structured_output


In [ ]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model


class Recipe(BaseModel):
    ingredients: list[str] = Field(description="ingredients of the dish")
    steps: list[str] = Field(description="steps to make the dish")


model = init_chat_model(
    model="gpt-5-nano",
    model_provider="openai",
    reasoning_effort="minimal",
)
structured_model = model.with_structured_output(Recipe)
result = structured_model.invoke("カレーのレシピを教えて")

print(type(result))
print(result)

## Q&Aアプリケーション


In [ ]:
ROLES = {
    1: {
        "name": "一般知識エキスパート",
        "description": "幅広い分野の一般的な質問に答える",
        "details": "幅広い分野の一般的な質問に対して、正確で分かりやすい回答を提供してください。",
    },
    2: {
        "name": "生成AI製品エキスパート",
        "description": "生成AIや関連製品、技術に関する専門的な質問に答える",
        "details": "生成AIや関連製品、技術に関する専門的な質問に対して、最新の情報と深い洞察を提供してください。",
    },
    3: {
        "name": "カウンセラー",
        "description": "個人的な悩みや心理的な問題に対してサポートを提供する",
        "details": "個人的な悩みや心理的な問題に対して、共感的で支援的な回答を提供し、可能であれば適切なアドバイスも行ってください。",
    },
}

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages


class State(TypedDict):
    query: str
    current_role: str
    messages: Annotated[list[BaseMessage], add_messages]
    current_judge: bool
    judgement_reason: str

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="gpt-5-mini",
    model_provider="openai",
    reasoning_effort="medium",
)

In [ ]:
from typing import Any

from pydantic import BaseModel


class SelectionOutput(BaseModel):
    role_number: int


selection_prompt_template = """
質問を分析し、最も適切な回答担当ロールを選択してください。

選択肢:
{role_options}

回答は選択肢の番号（1、2、または3）のみを返してください。

質問: {query}
""".strip()


def selection_node(state: State) -> dict[str, Any]:
    query = state["query"]
    role_options = "\n".join(
        [f"{k}. {v['name']}: {v['description']}" for k, v in ROLES.items()]
    )
    prompt = selection_prompt_template.format(role_options=role_options, query=query)

    model_with_structure = llm.with_structured_output(SelectionOutput)
    output: SelectionOutput = model_with_structure.invoke(prompt)

    selected_role = ROLES[output.role_number]["name"]
    return {"current_role": selected_role}

In [ ]:
answering_prompt_template = """
あなたは{role_name}として回答してください。以下の質問に対して、あなたの役割に基づいた適切な回答を提供してください。

役割の詳細:
{role_detail}

質問: {query}

回答:
""".strip()


def answering_node(state: State) -> dict[str, Any]:
    query = state["query"]
    role_name = state["current_role"]
    role = next(filter(lambda x: x["name"] == role_name, ROLES.values()))
    role_detail = role["details"]
    prompt = answering_prompt_template.format(
        role_name=role_name,
        role_detail=role_detail,
        query=query,
    )

    answer = llm.invoke(prompt)
    return {"messages": [answer]}


In [ ]:
from pydantic import BaseModel, Field


class Judgement(BaseModel):
    reason: str = Field(description="判定理由")
    judge: bool = Field(description="判定結果")


judgement_prompt_template = """
以下の回答の品質をチェックし、問題がある場合は'False'、問題がない場合は'True'を回答してください。
また、その判断理由も説明してください。

ユーザーからの質問: {query}
回答: {answer}
""".strip()


def check_node(state: State) -> dict[str, Any]:
    query = state["query"]
    answer = state["messages"][-1]
    prompt = judgement_prompt_template.format(query=query, answer=answer.content)

    model_with_structure = llm.with_structured_output(Judgement)
    result: Judgement = model_with_structure.invoke(prompt)

    return {"current_judge": result.judge, "judgement_reason": result.reason}

In [ ]:
from langgraph.graph import StateGraph, START, END

graph_builder = StateGraph(State)

graph_builder.add_node("selection", selection_node)
graph_builder.add_node("answering", answering_node)
graph_builder.add_node("check", check_node)

graph_builder.add_edge(START, "selection")
graph_builder.add_edge("selection", "answering")
graph_builder.add_edge("answering", "check")

# checkノードから次のノードへの遷移に条件付きエッジを定義
# state.current_judgeの値がTrueならENDノードへ、Falseならselectionノードへ
graph_builder.add_conditional_edges(
    "check", lambda state: state["current_judge"], {True: END, False: "selection"}
)

graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
initial_state = {
    "query": "生成AIについて教えてください",
    "current_role": "",
    "messages": [],
    "current_judge": False,
    "judgement_reason": "",
}
graph.invoke(initial_state)

# Tool useとAIエージェント


## Function calling


In [ ]:
import json


def get_current_weather(location: str, unit: str = "celsius") -> str:
    return json.dumps({"location": location, "temperature": 20, "unit": unit})

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city, e.g. Tokyo",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "default": "celsius",
                    },
                },
                "required": ["location"],
            },
        },
    }
]

In [ ]:
from openai import OpenAI

client = OpenAI()

messages = [
    {"role": "user", "content": "東京の現在の天気はどうですか？"},
]

response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
    reasoning_effort="minimal",
)
print(response.to_json(indent=2))

In [ ]:
response_message = response.choices[0].message
messages.append(response_message.to_dict())
print(json.dumps(messages, ensure_ascii=False, indent=2))

In [ ]:
available_functions = {
    "get_current_weather": get_current_weather,
}

# 使いたい関数は複数あるかもしれないのでループ
for tool_call in response_message.tool_calls:
    # 関数を実行
    function_name = tool_call.function.name
    function_to_call = available_functions[function_name]
    function_args = json.loads(tool_call.function.arguments)
    function_response = function_to_call(**function_args)
    print(function_response.encode("utf-8").decode("unicode_escape"))

    # 関数の実行結果を会話履歴としてmessagesに追加
    messages.append(
        {
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": function_name,
            "content": function_response,
        }
    )

print(json.dumps(messages, ensure_ascii=False, indent=2))

In [ ]:
second_response = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    reasoning_effort="minimal",
)
print(second_response.to_json(indent=2))